# Libraries

In [1]:
import pandas as pd
import numpy as np
import datetime as dt
import logging
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import talib as ta
from torch import optim
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split
from tqdm import tqdm
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.optim import AdamW
from utils import scale, inverse_scale, inverse_scale_pair, inspect
from utils.paths import CHECKPOINTS_DIR
from pypfopt import risk_models, expected_returns, plotting, EfficientFrontier

# Own Libs
from config import *
from entities import *
from strategies import *
from datasets import *
from engine import Engine
from models import DiffusionTransformer, Diffusion

# Setup

In [2]:
logging.basicConfig(level=logging.DEBUG)
logging.getLogger('matplotlib').setLevel(logging.WARNING)

In [3]:
cfg = TrainConfig(epochs=1000, window_size=64, steps_to_sim=12, num_sims=100)
window_size = cfg.window_size
device = cfg.device
batch_size = cfg.batch_size
epochs = cfg.epochs
sim_steps = cfg.steps_to_sim
num_sims = cfg.num_sims

# Optimizer
weight_decay = cfg.optimizer.weight_decay
lr = cfg.optimizer.lr

time_range = {
    'start_date': '2021-01-01',
    'end_date': '2024-12-31'
}

ddpm = {
    'timesteps': int(1000),
    'beta_start': 0.0001,
    'beta_end': 0.02
}

ddpm_transformer = {
    'window_size': window_size,
    'd_model': 64,
    'nhead': 4,
    'num_layers': 64,
    'dim_feedforward': 512,
    'dropout': 0.1
}

The history saving thread hit an unexpected error (OperationalError('database is locked')).History will not be written to the database.


# Data [N, W, A, F]
**[N, T, A, F]** means: 
* **N**: Num of Window or Num of Batch
* **W**: Window
* **A**: Assets
* **F**: Features or Channels

In [4]:
def time_range_info(df):
    info = (df.index.min(), df.index.max())
    print(f"Data range: {info[0]} to {info[1]}")
    
    duration = df.index.max() - df.index.min()
    print(f"Total duration: {duration}")

def time_range_mask(df, start_date, end_date):
    mask = (df.index >= start_date) & (df.index <= end_date)
    return mask

In [5]:
symbols = ['AAPL', 'TSLA', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'GOOG', 'META', 'AVGO', 'ORCL', 'CRM', 'ADBE', 'AMD', 'CSCO']
freq = "1d"

# Basket
basket = Basket(symbols=symbols)
basket.load_all_assets(freq=freq)

print(f"Basket data shape: {basket.data.shape}")
basket.data.head(5)

DEBUG:entities.basket:Initialized Asset Basket: ['AAPL', 'TSLA', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'GOOG', 'META', 'AVGO', 'ORCL', 'CRM', 'ADBE', 'AMD', 'CSCO'] with 0 assets which loaded.
INFO:entities.basket:Starting batch load for 14 symbols...
DEBUG:entities.basket:Attempting to load AAPL...
DEBUG:entities.asset:Initialized Asset: AAPL with 2724 rows.
INFO:entities.basket:Successfully loaded AAPL (2724 rows).
DEBUG:entities.basket:Attempting to load TSLA...
DEBUG:entities.asset:Initialized Asset: TSLA with 2760 rows.
INFO:entities.basket:Successfully loaded TSLA (2760 rows).
DEBUG:entities.basket:Attempting to load MSFT...
DEBUG:entities.asset:Initialized Asset: MSFT with 2724 rows.
INFO:entities.basket:Successfully loaded MSFT (2724 rows).
DEBUG:entities.basket:Attempting to load NVDA...
DEBUG:entities.asset:Initialized Asset: NVDA with 2724 rows.
INFO:entities.basket:Successfully loaded NVDA (2724 rows).
DEBUG:entities.basket:Attempting to load GOOGL...
DEBUG:entities.asset:Initia

Basket data shape: (2760, 70)


AAPL                                                \
                Close       High        Low       Open       Volume   
Date                                                                  
2015-01-02  24.261047  24.729270  23.821672  24.718174  212818400.0   
2015-01-05  23.577574  24.110150  23.391173  24.030263  257142000.0   
2015-01-06  23.579794  23.839424  23.218085  23.641928  263188400.0   
2015-01-07  23.910433  24.010290  23.677430  23.788384  160423600.0   
2015-01-08  24.829119  24.886815  24.121236  24.238848  237458000.0   

                 TSLA                                             ...   AMD  \
                Close       High        Low       Open    Volume  ... Close   
Date                                                              ...         
2015-01-02  14.620667  14.883333  14.217333  14.858000  71466000  ...  2.67   
2015-01-05  14.006000  14.433333  13.810667  14.303333  80527500  ...  2.66   
2015-01-06  14.085333  14.280000  13.614000  14.004000  93928500  ...  2.63   
2015-01-07  14.063333  14.318667  13.985333  14.223333  44526000  ...  2.58   
2015-01-08  14.041333  14.253333  14.000667  14.187333  51637500  ...  2.61   

                                               CSCO                        \
            High   Low  Open      Volume      Close       High        Low   
Date                                                                        
2015-01-02  2.67  2.67  2.67         0.0  19.815605  20.181631  19.650534   
2015-01-05  2.70  2.64  2.67   8878200.0  19.420874  19.700776  19.377812   
2015-01-06  2.66  2.55  2.65  13912500.0  19.413698  19.865848  19.406522   
2015-01-07  2.65  2.54  2.63  12377600.0  19.593126  19.664896  19.363463   
2015-01-08  2.65  2.56  2.59  11136600.0  19.743843  20.160107  19.715135   

                                   
                 Open      Volume  
Date                               
2015-01-02  19.995029  22926500.0  
2015-01-05  19.607475  29460600.0  
2015-01-06  19.478291  47297600.0  
2015-01-07  19.478295  27570800.0  
2015-01-08  19.765374  40907000.0  

[5 rows x 70 columns]

## Time Range Custom

In [6]:
time_range_info(basket.data)

for symbol, asset in basket.assets.items():
    mask = time_range_mask(asset.data, time_range['start_date'], time_range['end_date'])
    asset.data = asset.data[mask]

time_range_info(basket.data)

Data range: 2015-01-02 00:00:00 to 2025-12-22 00:00:00
Total duration: 4007 days 00:00:00
Data range: 2021-01-04 00:00:00 to 2024-12-31 00:00:00
Total duration: 1457 days 00:00:00


## Features/Channels ($F$)
1. Find Joint Distribution $F_{\text{date\ A}} \cap F_{\text{date\ B}}$ with intersection
2. Select $F$ to norm as Return values

In [7]:
targets = ["Close"]
features = basket.get_unique_features()
print(f"Features:\t{features}\nTargets:\t{targets}")

Features:	['Close', 'High', 'Low', 'Open', 'Volume']
Targets:	['Close']


In [8]:
print(f"Basket data shape before Joint: {basket.data.shape}")

joint_strategy = IntersectionStrategy()
basket.align(joint_strategy)

print(f"Basket data shape after Joint: {basket.data.shape}")

INFO:strategies.concrete:Aligned: 14 orig -> 14 clean assets -> 1005 rows
DEBUG:entities.basket:Aligned data shape: (1005, 70)
INFO:entities.basket:Assets updated in-place to aligned index (Length: 1005)


Basket data shape before Joint: (1005, 70)
Basket data shape after Joint: (1005, 70)


In [9]:
basket.to_returns(features=targets, log=True, keep=False)
targets = basket.get_keyword_features("Returns")
features = basket.get_unique_features()

print(f"Features:\t{features}\nTargets:\t{targets}")
basket.data.head(5)

DEBUG:entities.asset:AAPL converted to Returns (log=True)
DEBUG:entities.asset:TSLA converted to Returns (log=True)
DEBUG:entities.asset:MSFT converted to Returns (log=True)
DEBUG:entities.asset:NVDA converted to Returns (log=True)
DEBUG:entities.asset:GOOGL converted to Returns (log=True)
DEBUG:entities.asset:AMZN converted to Returns (log=True)
DEBUG:entities.asset:GOOG converted to Returns (log=True)
DEBUG:entities.asset:META converted to Returns (log=True)
DEBUG:entities.asset:AVGO converted to Returns (log=True)
DEBUG:entities.asset:ORCL converted to Returns (log=True)
DEBUG:entities.asset:CRM converted to Returns (log=True)
DEBUG:entities.asset:ADBE converted to Returns (log=True)
DEBUG:entities.asset:AMD converted to Returns (log=True)
DEBUG:entities.asset:CSCO converted to Returns (log=True)


Features:	['Close_Log_Returns', 'High', 'Low', 'Open', 'Volume']
Targets:	{'Close_Log_Returns'}


AAPL                                                       \
                  High         Low        Open     Volume Close_Log_Returns   
Date                                                                          
2021-01-05  128.366929  125.141666  125.589894   97664900          0.012288   
2021-01-06  127.694587  123.144152  124.449847  155088000         -0.034241   
2021-01-07  128.259768  124.586290  125.073488  109578200          0.033554   
2021-01-08  129.234127  126.895568  129.039236  105158200          0.008594   
2021-01-11  126.837115  125.209876  125.882211  100384500         -0.023523   

                  TSLA                                                       \
                  High         Low        Open     Volume Close_Log_Returns   
Date                                                                          
2021-01-05  246.946671  239.733337  241.220001   96735600          0.007291   
2021-01-06  258.000000  249.699997  252.830002  134100000          0.027995   
2021-01-07  272.329987  258.399994  259.209991  154496700          0.076448   
2021-01-08  294.829987  279.463318  285.333344  225166500          0.075481   
2021-01-11  284.809998  267.873322  283.133331  177904800         -0.081442   

            ...        AMD                                                    \
            ...       High        Low       Open    Volume Close_Log_Returns   
Date        ...                                                                
2021-01-05  ...  93.209999  91.410004  92.099998  34208000          0.005079   
2021-01-06  ...  92.279999  89.459999  91.620003  51911700         -0.026654   
2021-01-07  ...  95.510002  91.199997  91.330002  42897200          0.052090   
2021-01-08  ...  96.400002  93.269997  95.980003  39816400         -0.006114   
2021-01-11  ...  99.230003  93.760002  94.029999  48600200          0.027839   

                 CSCO                                                    
                 High        Low       Open    Volume Close_Log_Returns  
Date                                                                     
2021-01-05  38.335017  37.734811  37.995770  17763700          0.000455  
2021-01-06  39.030909  38.178440  38.387210  21823100          0.009505  
2021-01-07  39.239677  38.422000  38.448099  18218800          0.012534  
2021-01-08  39.500638  38.491593  38.691662  20936300          0.002222  
2021-01-11  39.970367  39.161391  39.274475  25058200          0.006636  

[5 rows x 70 columns]

## Add Indicators as Features ($F$)

In [10]:
# Indicator
time_prd = 20
fast_prd, slow_prd, signal_prd = 12, 26, 9

for symbol, asset in basket.assets.items():
    df = asset.data 
    
    for target in targets:
        s = df[target]
        
        df[f"SMA_{time_prd} {target}"] = ta.SMA(s, timeperiod=time_prd)
        df[f"EMA_{time_prd} {target}"] = ta.EMA(s, timeperiod=time_prd)
        df[f"RSI_{time_prd} {target}"] = ta.RSI(s, timeperiod=time_prd)
        
        macd, signal, hist = ta.MACD(s, fastperiod=fast_prd, slowperiod=slow_prd, signalperiod=signal_prd)
        df[f"MACD {target}"] = macd
        df[f"MACD_Sig {target}"] = signal
        df[f"MACD_Hist {target}"] = hist

print(f"Basket data shape: {basket.data.shape}")
basket.data.head(5)

Basket data shape: (1004, 154)


AAPL                                                       \
                  High         Low        Open     Volume Close_Log_Returns   
Date                                                                          
2021-01-05  128.366929  125.141666  125.589894   97664900          0.012288   
2021-01-06  127.694587  123.144152  124.449847  155088000         -0.034241   
2021-01-07  128.259768  124.586290  125.073488  109578200          0.033554   
2021-01-08  129.234127  126.895568  129.039236  105158200          0.008594   
2021-01-11  126.837115  125.209876  125.882211  100384500         -0.023523   

                                                              \
           SMA_20 Close_Log_Returns EMA_20 Close_Log_Returns   
Date                                                           
2021-01-05                      NaN                      NaN   
2021-01-06                      NaN                      NaN   
2021-01-07                      NaN                      NaN   
2021-01-08                      NaN                      NaN   
2021-01-11                      NaN                      NaN   

                                                            \
           RSI_20 Close_Log_Returns MACD Close_Log_Returns   
Date                                                         
2021-01-05                      NaN                    NaN   
2021-01-06                      NaN                    NaN   
2021-01-07                      NaN                    NaN   
2021-01-08                      NaN                    NaN   
2021-01-11                      NaN                    NaN   

                                       ...       CSCO                       \
           MACD_Sig Close_Log_Returns  ...        Low       Open    Volume   
Date                                   ...                                   
2021-01-05                        NaN  ...  37.734811  37.995770  17763700   
2021-01-06                        NaN  ...  38.178440  38.387210  21823100   
2021-01-07                        NaN  ...  38.422000  38.448099  18218800   
2021-01-08                        NaN  ...  38.491593  38.691662  20936300   
2021-01-11                        NaN  ...  39.161391  39.274475  25058200   

                                                       \
           Close_Log_Returns SMA_20 Close_Log_Returns   
Date                                                    
2021-01-05          0.000455                      NaN   
2021-01-06          0.009505                      NaN   
2021-01-07          0.012534                      NaN   
2021-01-08          0.002222                      NaN   
2021-01-11          0.006636                      NaN   

                                                              \
           EMA_20 Close_Log_Returns RSI_20 Close_Log_Returns   
Date                                                           
2021-01-05                      NaN                      NaN   
2021-01-06                      NaN                      NaN   
2021-01-07                      NaN                      NaN   
2021-01-08                      NaN                      NaN   
2021-01-11                      NaN                      NaN   

                                                              \
           MACD Close_Log_Returns MACD_Sig Close_Log_Returns   
Date                                                           
2021-01-05                    NaN                        NaN   
2021-01-06                    NaN                        NaN   
2021-01-07                    NaN                        NaN   
2021-01-08                    NaN                        NaN   
2021-01-11                    NaN                        NaN   

                                        
           MACD_Hist Close_Log_Returns  
Date                                    
2021-01-05                         NaN  
2021-01-06                         NaN  
2021-01-07                         NaN  
2021-01-08           

## Additional! Shift data for future simulation (forward)

In [11]:
for symbol, asset in basket.assets.items():
    df = asset.data 
    
    for target in targets:
        df[f"SMA_{time_prd} {target}"] = df[f"SMA_{time_prd} {target}"].shift(1)
        df[f"EMA_{time_prd} {target}"] = df[f"EMA_{time_prd} {target}"].shift(1)
        df[f"RSI_{time_prd} {target}"] = df[f"RSI_{time_prd} {target}"].shift(1)
        df[f"MACD {target}"] = df[f"MACD {target}"].shift(1)
        df[f"MACD_Sig {target}"] = df[f"MACD_Sig {target}"].shift(1)
        df[f"MACD_Hist {target}"] = df[f"MACD_Hist {target}"].shift(1)

    asset.data = df.dropna() 
    
print(f"Basket data shape: {basket.data.shape}")
basket.data.head(5)

Basket data shape: (970, 154)


AAPL                                                       \
                  High         Low        Open     Volume Close_Log_Returns   
Date                                                                          
2021-02-24  122.527972  119.278390  121.922948  111039900         -0.004060   
2021-02-25  123.406232  117.629191  121.669217  148199500         -0.035402   
2021-02-26  121.835107  118.273246  119.629680  164560400          0.002229   
2021-03-01  124.840766  119.824887  120.761704  116307900          0.052452   
2021-03-02  125.611668  121.991258  125.309156  102260900         -0.021115   

                                                              \
           SMA_20 Close_Log_Returns EMA_20 Close_Log_Returns   
Date                                                           
2021-02-24                -0.006281                -0.004764   
2021-02-25                -0.006568                -0.004697   
2021-02-26                -0.007952                -0.007621   
2021-03-01                -0.006060                -0.006683   
2021-03-02                -0.001531                -0.001051   

                                                            \
           RSI_20 Close_Log_Returns MACD Close_Log_Returns   
Date                                                         
2021-02-24                49.443327              -0.005006   
2021-02-25                49.042061              -0.004288   
2021-02-26                44.959599              -0.006177   
2021-03-01                50.199121              -0.004584   
2021-03-02                56.073523               0.000722   

                                       ...       CSCO                       \
           MACD_Sig Close_Log_Returns  ...        Low       Open    Volume   
Date                                   ...                                   
2021-02-24                  -0.005451  ...  39.178786  39.352760  17823600   
2021-02-25                  -0.005218  ...  39.352752  39.657204  21916700   
2021-02-26                  -0.005410  ...  38.935217  39.648511  22144900   
2021-03-01                  -0.005245  ...  39.335356  39.335356  17394100   
2021-03-02                  -0.004051  ...  39.509325  39.952959  14833000   

                                                       \
           Close_Log_Returns SMA_20 Close_Log_Returns   
Date                                                    
2021-02-24          0.005041                 0.000530   
2021-02-25         -0.004822                 0.000527   
2021-02-26         -0.014382                -0.000197   
2021-03-01          0.023131                -0.000521   
2021-03-02         -0.008749                 0.001481   

                                                              \
           EMA_20 Close_Log_Returns RSI_20 Close_Log_Returns   
Date                                                           
2021-02-24                -0.001497                50.527952   
2021-02-25                -0.000874                51.224838   
2021-02-26                -0.001250                49.039717   
2021-03-01                -0.002501                46.994221   
2021-03-02                -0.000060                54.783902   

                                                              \
           MACD Close_Log_Returns MACD_Sig Close_Log_Returns   
Date                                                           
2021-02-24              -0.001681                  -0.000759   
2021-02-25              -0.000967                  -0.000801   
2021-02-26              -0.001184                  -0.000877   
2021-03-01              -0.002102                  -0.001122   
2021-03-02               0.000194                  -0.000859   

                                        
           MACD_Hist Close_Log_Returns  
Date                                    
2021-02-24                   -0.000923  
2021-02-25                   -0.000167  
2021-02-26                   -0.000307  
2021-03-01           

In [12]:
basket.align(joint_strategy)

print(f"Basket data shape: {basket.data.shape}")
basket.data.head(5)

INFO:strategies.concrete:Aligned: 14 orig -> 14 clean assets -> 970 rows
DEBUG:entities.basket:Aligned data shape: (970, 154)
INFO:entities.basket:Assets updated in-place to aligned index (Length: 970)


Basket data shape: (970, 154)


AAPL                                                       \
                  High         Low        Open     Volume Close_Log_Returns   
Date                                                                          
2021-02-24  122.527972  119.278390  121.922948  111039900         -0.004060   
2021-02-25  123.406232  117.629191  121.669217  148199500         -0.035402   
2021-02-26  121.835107  118.273246  119.629680  164560400          0.002229   
2021-03-01  124.840766  119.824887  120.761704  116307900          0.052452   
2021-03-02  125.611668  121.991258  125.309156  102260900         -0.021115   

                                                              \
           SMA_20 Close_Log_Returns EMA_20 Close_Log_Returns   
Date                                                           
2021-02-24                -0.006281                -0.004764   
2021-02-25                -0.006568                -0.004697   
2021-02-26                -0.007952                -0.007621   
2021-03-01                -0.006060                -0.006683   
2021-03-02                -0.001531                -0.001051   

                                                            \
           RSI_20 Close_Log_Returns MACD Close_Log_Returns   
Date                                                         
2021-02-24                49.443327              -0.005006   
2021-02-25                49.042061              -0.004288   
2021-02-26                44.959599              -0.006177   
2021-03-01                50.199121              -0.004584   
2021-03-02                56.073523               0.000722   

                                       ...       CSCO                       \
           MACD_Sig Close_Log_Returns  ...        Low       Open    Volume   
Date                                   ...                                   
2021-02-24                  -0.005451  ...  39.178786  39.352760  17823600   
2021-02-25                  -0.005218  ...  39.352752  39.657204  21916700   
2021-02-26                  -0.005410  ...  38.935217  39.648511  22144900   
2021-03-01                  -0.005245  ...  39.335356  39.335356  17394100   
2021-03-02                  -0.004051  ...  39.509325  39.952959  14833000   

                                                       \
           Close_Log_Returns SMA_20 Close_Log_Returns   
Date                                                    
2021-02-24          0.005041                 0.000530   
2021-02-25         -0.004822                 0.000527   
2021-02-26         -0.014382                -0.000197   
2021-03-01          0.023131                -0.000521   
2021-03-02         -0.008749                 0.001481   

                                                              \
           EMA_20 Close_Log_Returns RSI_20 Close_Log_Returns   
Date                                                           
2021-02-24                -0.001497                50.527952   
2021-02-25                -0.000874                51.224838   
2021-02-26                -0.001250                49.039717   
2021-03-01                -0.002501                46.994221   
2021-03-02                -0.000060                54.783902   

                                                              \
           MACD Close_Log_Returns MACD_Sig Close_Log_Returns   
Date                                                           
2021-02-24              -0.001681                  -0.000759   
2021-02-25              -0.000967                  -0.000801   
2021-02-26              -0.001184                  -0.000877   
2021-03-01              -0.002102                  -0.001122   
2021-03-02               0.000194                  -0.000859   

                                        
           MACD_Hist Close_Log_Returns  
Date                                    
2021-02-24                   -0.000923  
2021-02-25                   -0.000167  
2021-02-26                   -0.000307  
2021-03-01           

## Filter only Target Features ($F_{target} $)

In [13]:
targets = basket.get_keyword_features("Returns")
print(f"Targets: {targets}")


for symbol, asset in basket.assets.items():
    mask = asset.data.columns.isin(targets)
    asset.data = asset.data.loc[:, mask]

print(f"Basket shape: {basket.data.shape}")
basket.data.head(5)

Targets: {'Close_Log_Returns', 'MACD_Hist Close_Log_Returns', 'EMA_20 Close_Log_Returns', 'RSI_20 Close_Log_Returns', 'SMA_20 Close_Log_Returns', 'MACD_Sig Close_Log_Returns', 'MACD Close_Log_Returns'}
Basket shape: (970, 98)


AAPL                           \
           Close_Log_Returns SMA_20 Close_Log_Returns   
Date                                                    
2021-02-24         -0.004060                -0.006281   
2021-02-25         -0.035402                -0.006568   
2021-02-26          0.002229                -0.007952   
2021-03-01          0.052452                -0.006060   
2021-03-02         -0.021115                -0.001531   

                                                              \
           EMA_20 Close_Log_Returns RSI_20 Close_Log_Returns   
Date                                                           
2021-02-24                -0.004764                49.443327   
2021-02-25                -0.004697                49.042061   
2021-02-26                -0.007621                44.959599   
2021-03-01                -0.006683                50.199121   
2021-03-02                -0.001051                56.073523   

                                                              \
           MACD Close_Log_Returns MACD_Sig Close_Log_Returns   
Date                                                           
2021-02-24              -0.005006                  -0.005451   
2021-02-25              -0.004288                  -0.005218   
2021-02-26              -0.006177                  -0.005410   
2021-03-01              -0.004584                  -0.005245   
2021-03-02               0.000722                  -0.004051   

                                                    TSLA  \
           MACD_Hist Close_Log_Returns Close_Log_Returns   
Date                                                       
2021-02-24                    0.000445          0.059954   
2021-02-25                    0.000930         -0.084024   
2021-02-26                   -0.000767         -0.009899   
2021-03-01                    0.000661          0.061615   
2021-03-02                    0.004773         -0.045549   

                                                              ...  \
           SMA_20 Close_Log_Returns EMA_20 Close_Log_Returns  ...   
Date                                                          ...   
2021-02-24                -0.011570                -0.012856  ...   
2021-02-25                -0.008703                -0.005921  ...   
2021-02-26                -0.011820                -0.013360  ...   
2021-03-01                -0.010625                -0.013030  ...   
2021-03-02                -0.004971                -0.005921  ...   

                              AMD                             \
           MACD Close_Log_Returns MACD_Sig Close_Log_Returns   
Date                                                           
2021-02-24              -0.004678                  -0.002595   
2021-02-25              -0.001476                  -0.002371   
2021-02-26              -0.005253                  -0.002947   
2021-03-01              -0.001896                  -0.002737   
2021-03-02               0.000513                  -0.002087   

                                                    CSCO  \
           MACD_Hist Close_Log_Returns Close_Log_Returns   
Date                                                       
2021-02-24                   -0.002084          0.005041   
2021-02-25                    0.000895         -0.004822   
2021-02-26                   -0.002306         -0.014382   
2021-03-01                    0.000841          0.023131   
2021-03-02                    0.002600         -0.008749   

                                                              \
           SMA_20 Close_Log_Returns EMA_20 Close_Log_Returns   
Date                                                           
2021-02-24                 0.000530                -0.001497   
2021-02-25                 0.000527                -0.000874   
2021-02-26                -0.000197                -0.001250   
2021-03-01                -0.000521                -0.002501   
2021-03-02                 0.001481                -0.000060   



# Dataset & Dataloader

In [14]:
n_obs = len(basket.data)
n_assets = basket.data.columns.levels[0].size
n_features = basket.data.columns.levels[1].size

print(n_obs, n_assets, n_features)

basket_np = basket.data.values.reshape(n_obs, n_assets, n_features)
basket_np.shape

970 14 7


(970, 14, 7)

## Ratio Dataset

In [15]:
ratios = [0.8, 0.1, 0.1]
total_count = len(basket_np)
train_count = int(total_count * ratios[0])
val_count = int(total_count * ratios[1])
test_count = total_count - train_count - val_count

print(f"Ratios DS\nTrain:\t{train_count}\nVal:\t{val_count}\nTest:\t{test_count}\nTotal:\t{total_count}")

Ratios DS
Train:	776
Val:	97
Test:	97
Total:	970


In [16]:
all_dates = basket.data.index.get_level_values(0).unique().to_numpy()
print(f"Dates shape: {all_dates.shape}")

Dates shape: (970,)


In [17]:
end_val = train_count + val_count

# Ratios
train_part = basket_np[:train_count]
val_part = basket_np[train_count:end_val]
test_part = basket_np[end_val:]

train_dates = all_dates[:train_count]
val_dates   = all_dates[train_count:end_val]
test_dates  = all_dates[end_val:]

print(f"Train: {train_part.shape}\nVal: {val_part.shape}\nTest:{test_part.shape}")
print(f"Train Dates: {train_dates.shape}\nVal Dates: {val_dates.shape}\nTest Dates:{test_dates.shape}")

Train: (776, 14, 7)
Val: (97, 14, 7)
Test:(97, 14, 7)
Train Dates: (776,)
Val Dates: (97,)
Test Dates:(97,)


## Scale Dataset

In [18]:
# scaler = MinMaxScaler(feature_range=(-1, 1))
scaler = StandardScaler()

# Require 2D Numpy Array
T, A, F = train_part.shape
scaler.fit(train_part.reshape(-1, F))

scaled_train_part = scale(train_part, scaler)
scaled_val_part = scale(val_part, scaler)
scaled_test_part = scale(test_part, scaler)

inspect(scaled_train_part, "Scaled Train Part")
inspect(scaled_val_part, "Scaled Val Part")
inspect(scaled_test_part, "Scaled Test Part")
print(f"Train:\t{scaled_train_part.shape}\nVal:\t{scaled_val_part.shape}\nTest:\t{scaled_test_part.shape}")

--- Inspecting: Scaled Train Part ---
------------------------------------
Shape: (776, 14, 7)
Min:   -12.4922
Max:   8.8458
Mean:  0.0000
Std:   1.0000
------------------------------------
--- Inspecting: Scaled Val Part ---
------------------------------------
Shape: (97, 14, 7)
Min:   -8.9719
Max:   6.1577
Mean:  -0.0437
Std:   0.9942
------------------------------------
--- Inspecting: Scaled Test Part ---
------------------------------------
Shape: (97, 14, 7)
Min:   -6.1881
Max:   8.8664
Mean:  0.1223
Std:   0.9497
------------------------------------
Train:	(776, 14, 7)
Val:	(97, 14, 7)
Test:	(97, 14, 7)


## Dataloader

In [19]:
train_ds = MarketDataset(data=scaled_train_part, dates=train_dates, window_size=window_size)
val_ds = MarketDataset(data=scaled_val_part, dates=val_dates, window_size=window_size)
test_ds = MarketDataset(data=scaled_test_part, dates=test_dates, window_size=window_size)

print(f"Num of Windows\nTrain DS: {len(train_ds)}, Val Ds: {len(val_ds)}, Test DS: {len(test_ds)}\n")
print(f"A sample shape from Train DS\n\tx: {train_ds[0]['x'].shape},\n\tx_cond: {train_ds[0]['x_cond'].shape}")

Num of Windows
Train DS: 713, Val Ds: 34, Test DS: 34

A sample shape from Train DS
	x: (64, 14, 1),
	x_cond: (64, 14, 6)


In [20]:
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

batch = next(iter(train_loader))
print(len(train_loader))
print(batch["x"].shape)
print(batch["x_cond"].shape)

90
torch.Size([8, 64, 14, 1])
torch.Size([8, 64, 14, 6])


# Model, Engine
Use *Condition DDPM* 

In [21]:
# n_window mean batch size
n_window, window, n_assets, n_features = batch["x"].shape
n_window, window, n_assets, n_conds = batch["x_cond"].shape
ddpm_transformer['n_cond'] = n_conds

print(n_assets, n_features, n_conds)

input_channels = n_assets * n_features
cond_channels = n_assets * n_conds
print(input_channels, cond_channels)

14 1 6
14 84


In [22]:
model = DiffusionTransformer(
    n_features=input_channels,
    n_cond=cond_channels,        
    window_size=window_size,             
    d_model=ddpm_transformer['d_model'],                
    nhead=ddpm_transformer['nhead'],
    num_layers=ddpm_transformer['num_layers'],
    dim_feedforward=ddpm_transformer['dim_feedforward'],
    dropout=ddpm_transformer['dropout']
).to(device)

In [23]:
diffusion = Diffusion(model, timesteps=ddpm['timesteps'], beta_start=ddpm['beta_start'], beta_end=ddpm['beta_end']).to(device)

In [24]:
optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

In [25]:
engine = Engine(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    model=diffusion,
    scaler=scaler,
    optimizer=optimizer,
    device=device,
    file_name=f"ddpm_transformer_d{ddpm_transformer['d_model']}_l{ddpm_transformer['num_layers']}",
)

In [26]:
engine.fit(epochs)

INFO:engine.trainer:Engine started Training for 1000 epochs on cuda...
Epoch 1/1000 [Val]: 100%|██████████| 5/5 [00:00<00:00, 27.95it/s]


End of Epoch 1 | Train Loss: 1.020402 | Val Loss: 1.023451
New Best Model Saved (Val Loss: 1.023451)


Epoch 2/1000 [Val]: 100%|██████████| 5/5 [00:00<00:00, 28.65it/s]


End of Epoch 2 | Train Loss: 1.005711 | Val Loss: 0.979984
New Best Model Saved (Val Loss: 0.979984)


Epoch 3/1000 [Val]: 100%|██████████| 5/5 [00:00<00:00, 28.43it/s]


...

End of Epoch 998 | Train Loss: 0.998591 | Val Loss: 0.999061


Epoch 999/1000 [Val]: 100%|██████████| 5/5 [00:00<00:00, 28.48it/s]


End of Epoch 999 | Train Loss: 1.000880 | Val Loss: 0.993612


Epoch 1000/1000 [Val]: 100%|██████████| 5/5 [00:00<00:00, 28.24it/s]


End of Epoch 1000 | Train Loss: 1.000991 | Val Loss: 0.991735


In [27]:
batch = next(iter(test_loader))
print(f"x: {batch['x'].shape}, x_cond: {batch['x_cond'].shape}, dates: {batch['dates'].shape}")

x: torch.Size([1, 64, 14, 1]), x_cond: torch.Size([1, 64, 14, 6]), dates: torch.Size([1, 64])


In [28]:
def transform_dates(dates: torch.Tensor) -> pd.DatetimeIndex:
    dates_np = dates.cpu().numpy()
    dates_pd = pd.to_datetime(dates_np.flatten(), unit='ns')
    return dates_pd

In [29]:
transform_dates(batch['dates'])

DatetimeIndex(['2024-08-14', '2024-08-15', '2024-08-16', '2024-08-19',
               '2024-08-20', '2024-08-21', '2024-08-22', '2024-08-23',
               '2024-08-26', '2024-08-27', '2024-08-28', '2024-08-29',
               '2024-08-30', '2024-09-03', '2024-09-04', '2024-09-05',
               '2024-09-06', '2024-09-09', '2024-09-10', '2024-09-11',
               '2024-09-12', '2024-09-13', '2024-09-16', '2024-09-17',
               '2024-09-18', '2024-09-19', '2024-09-20', '2024-09-23',
               '2024-09-24', '2024-09-25', '2024-09-26', '2024-09-27',
               '2024-09-30', '2024-10-01', '2024-10-02', '2024-10-03',
               '2024-10-04', '2024-10-07', '2024-10-08', '2024-10-09',
               '2024-10-10', '2024-10-11', '2024-10-14', '2024-10-15',
               '2024-10-16', '2024-10-17', '2024-10-18', '2024-10-21',
               '2024-10-22', '2024-10-23', '2024-10-24', '2024-10-25',
               '2024-10-28', '2024-10-29', '2024-10-30', '2024-10-31',
      

In [30]:
def autoregressive(x: np.ndarray, x_cond: np.ndarray, steps: int, n_samples: int):
    B, W, A, F_target = x.shape
    period_curr = W - steps
    device = next(engine.model.parameters()).device

    x_curr = torch.as_tensor(x[:, :period_curr, :, :], dtype=torch.float32).to(device)
    x_cond_curr = torch.as_tensor(x_cond[:, :period_curr, :, :], dtype=torch.float32).to(device)

    x_curr = x_curr.to(device)
    x_cond_curr = x_cond_curr.to(device)

    # Monte Carlo Build
    x_curr = x_curr.repeat(n_samples, 1, 1, 1)
    x_cond_curr = x_cond_curr.repeat(n_samples, 1, 1, 1)

    results = []
    print(f"Start Auto-regression: Initial W={period_curr}, Target Steps={steps}")

    pbar = tqdm(range(steps))
    for i in pbar:
        pbar.set_description(f"window size {x_curr.shape[1]}...")
        batch = {
            "x": x_curr,
            "x_cond": x_cond_curr
        }

        _, next_step_x, _, _ = engine.simulate(batch, steps=1, inverse_scale=False)
        
        results.append(next_step_x)
        
        x_curr = torch.cat([x_curr, next_step_x], dim=1)
            
        idx = period_curr + i
            
        next_cond_slice_np = x_cond[:, idx : idx+1, :, :]
            
        next_cond_slice = torch.as_tensor(next_cond_slice_np, dtype=torch.float32).to(device)
        next_cond_slice = next_cond_slice.repeat(n_samples, 1, 1, 1)

        x_cond_curr = torch.cat([x_cond_curr, next_cond_slice], dim=1)

    scaled_sim_genai = torch.cat(results, dim=1).cpu().numpy()
    sim_genai, sim_genai_cond = inverse_scale_pair(scaled_sim_genai, x_cond_curr[:, -steps:, : ,:], scaler)
    
    scaled_full_sim_genai = x_curr.detach().cpu().numpy()
    full_sim_genai, full_sim_genai_cond = inverse_scale_pair(scaled_full_sim_genai, x_cond_curr, scaler)
    
    return full_sim_genai, sim_genai, full_sim_genai_cond, sim_genai_cond

In [ ]:
full_sim_genai, sim_genai, _, _ = autoregressive(batch['x'], batch['x_cond'], 8, 1000)
print(f"x_curr: {x_curr.shape}, final_prediction: {final_prediction.shape}")

Start Auto-regression: Initial W=56, Target Steps=8


window size 60...:  50%|█████     | 4/8 [10:50<10:55, 163.90s/it]

In [ ]:
gt, gt_cond = inverse_scale_pair(batch['x'], batch['x_cond'], scaler)
stats_batch = {
    'x': torch.as_tensor(gt).to(device),
    'x_cond': torch.as_tensor(gt_cond).to(device)
}
sim_stats = engine.gbm_simulate(stats_batch, steps=8, n_samples=1000)
sim_stats.shape

In [ ]:
rand_ind = np.random.randint(0, len(sim_genai))
sim_genai_sample = sim_genai[rand_ind].squeeze(-1)
sim_stats_sample = sim_stats[rand_ind].squeeze(-1)
print(f"sim_genai_sample: {sim_genai_sample.shape}, sim_stats_sample: {sim_stats_sample.shape}")

In [ ]:
dates = transform_dates(batch['dates'])
dates.shape

In [ ]:
engine.benchmark(gt=gt, sim_genai=sim_genai_sample, sim_stats=sim_stats_sample, dates=dates)